In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

!gdown '1u2Q5cS0qCt0OEdtCzBcn4UfBKQAFMl30'


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../DATA/Telco-Customer-Churn.csv')

# **EDA**

In [ ]:
df.head()

In [ ]:
df.tail(8)

In [ ]:
df.sample(4)

In [ ]:
df.shape

In [ ]:
df.index

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.info()

In [ ]:
# Check for missing values
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.isna()

In [ ]:
df.isna().any(axis=0)

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
# Fill missing TotalCharges with tenure * MonthlyCharges

df.loc[df['TotalCharges'].isna(), 'TotalCharges'] = df['tenure'] * df['MonthlyCharges']

In [ ]:
# Check again for missing values
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.loc[5:9, 'TotalCharges']

In [ ]:
df['gender'].value_counts()

In [ ]:
df.head(1)

In [ ]:
# Convert categorical variables to numeric using mapping
df['Partner'] = df['Partner'].map({ 'No' : 0, 'Yes': 1})
df['Dependents'] = df['Dependents'].map({ 'No' : 0, 'Yes': 1})
df['PhoneService'] = df['PhoneService'].map({ 'No' : 0, 'Yes': 1})
df['MultipleLines'] = df['MultipleLines'].map({ 'No' : 0, 'Yes': 1, 'No phone service': 2})
df['InternetService'] = df['InternetService'].map({ 'No' : 0, 'DSL': 1, 'Fiber optic': 2})
df['OnlineSecurity'] = df['OnlineSecurity'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['OnlineBackup'] = df['OnlineBackup'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['DeviceProtection'] = df['DeviceProtection'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['TechSupport'] = df['TechSupport'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['StreamingTV'] = df['StreamingTV'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['StreamingMovies'] = df['StreamingMovies'].map({ 'No' : 0, 'Yes': 1, 'No internet service': 2})
df['Contract'] = df['Contract'].map({ 'Month-to-month' : 0, 'One year': 1, 'Two year': 2})
df['PaperlessBilling'] = df['PaperlessBilling'].map({ 'No' : 0, 'Yes': 1})
df['PaymentMethod'] = df['PaymentMethod'].map({ 'Electronic check' : 0, 'Mailed check': 1, 'Bank transfer (automatic)': 2, 'Credit card (automatic)': 3})
df['Churn'] = df['Churn'].map({ 'No' : 0, 'Yes': 1})


In [ ]:
df.head(1)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
# Get list of categorical columns (excluding customerID)
categorical_col_list = df.select_dtypes(include='object').columns.tolist()[1:]

In [ ]:
for i in categorical_col_list:
  print(f'Value counts for {i}: \n {df[i].value_counts()}')
  print('=============================')

In [ ]:
df.describe()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# get list of numeric columns
numeric_col_list = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
numeric_col_list

In [ ]:
for i in numeric_col_list:
  plt.figure()
  sns.boxplot(df[i])
  plt.title(f'Distribution of {i}')
  plt.show()

In [ ]:
for i in numeric_col_list:
  plt.figure()
  sns.histplot(df[i], kde=True)
  plt.title(f'Distribution of {i}')
  plt.show()

In [ ]:
for i in categorical_col_list:
  plt.figure()
  sns.countplot(df[i])
  plt.title(f'Distribution of {i}')
  plt.show()

# **Feature Engineering**

In [ ]:
df

In [ ]:
df.info()

In [ ]:
X = df.iloc[:, 1:-1]

In [ ]:
X

In [ ]:
y = df.iloc[:, -1]

In [ ]:
y

In [ ]:
y = y.map({'Yes': 1, 'No': 0})

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
numeric_col_list = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_list = X.select_dtypes(include='object').columns.tolist()

print(numeric_col_list)
print(categorical_col_list)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_col_list),
        ('cat', categorical_transformer, categorical_cols_list)
    ]
)

In [ ]:
preprocessor

# **Modeling**

In [ ]:
log_reg_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression())
    ]
)

In [ ]:
log_reg_pipeline.fit(X_train, y_train)

# **Model Evaluation**

In [ ]:
y_pred_log_reg = log_reg_pipeline.predict(X_test)

In [ ]:
cm=confusion_matrix(y_test, y_pred_log_reg)
print(classification_report(y_test, y_pred_log_reg))
print(cm)
print("Accuracy:", f"{accuracy_score(y_test, y_pred_log_reg) * 100:.2f} %")

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier())
    ]
)

In [ ]:
rf_param_grid = {
    'classifier__n_estimators': [i for i in range(100, 1001 , 100)],
    'classifier__max_depth': [None, 5,10, 15, 20],
    'classifier__min_samples_split': [2, 5, 7,10],    
    'classifier__min_samples_leaf': [1, 2, 5,10]   
}


In [ ]:
grid_search_rf = GridSearchCV(
    estimator=rf_pipeline, 
    param_grid=rf_param_grid, 
    scoring='f1',
    cv=5, n_jobs=-1, verbose=1)

In [ ]:

grid_search_rf.fit(X_train, y_train)

In [ ]:
print("Best Hyperparameters for Random Forest:", grid_search_rf.best_params_)

In [ ]:
best_rf = grid_search_rf.best_estimator_

y_pred_rf= best_rf.predict(X_test)

In [ ]:
cm=confusion_matrix(y_test, y_pred_rf)
print(classification_report(y_test, y_pred_rf))
print(cm)
print("Accuracy:", f"{accuracy_score(y_test, y_pred_rf) * 100:.2f} %")

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
xgboost_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
    ]
)

In [ ]:
xgboost_param_grid = {
    'classifier__n_estimators': [i for i in range(100, 1001 , 100)],
    'classifier__max_depth': [None, 5,10, 15, 20],
    'classifier__learning_rate': [0.01, 0.1, 0.2, 0.3   ],  
    'classifier__subsample': [0.6, 0.8, 1.0],
    'classifier__colsample_bytree': [0.6, 0.8, 1.0],
    'classifier__gamma': [0, 0.1, 0.2, 0.3],
}

In [ ]:
xgboost_grid_search = GridSearchCV(
    estimator=xgboost_pipeline, 
    param_grid=xgboost_param_grid, 
    scoring='f1',
    cv=5, n_jobs=-1, verbose=1)


In [ ]:
!pip install joblib

In [ ]:
import joblib

In [ ]:
joblib.dump(log_reg_pipeline, 'log_reg_pipeline.joblib')
print   ('Logistic Regression pipeline saved successfully!')